# Лабораторная работа 1
### Вариант 4

Выполнили:
> Галеев Егор Вячеславович, J3119, 505106

> Образцов Максим Евгеньевич, J3119, 466930

**Исходные данные**

- `housing.csv` - данные о ценах о жилье в Бостоне, где 
- `MEDV` - медианная стоимость жилья в (в тыс. $)
- Признаки (13 столбцов): криминальная обстановка, доля жилых зон, расстояние
до центров занятости, количество комнат и др.
- `test_size` = 0.27
- `random_state` = 40 
---

### 1. Цель работы 
Предложить среднеквадратическую аппроксимацию табличной функции многих переменных, проанализировать чувствительность точного решения к ошибкам округления,
проверить сходимость расчетных и исходных данных

### 2. Теоретические сведения

**Линейная регрессия** — это метод моделирования зависимости между зависимой переменной $y$ и одной или несколькими независимыми переменными $X$:

  $$y=Xw+\epsilon$$
  где $w$ — вектор весов (коэффициентов), а $\epsilon$ — случайная ошибка.  
**Задача обучения** заключается в поиске оптимальных весов $w$, которые минимизируют функцию потерь.
Для среднеквадратичной ошибки (MSE) решение записывается в виде:
$$w^{*}=\arg\min_{w}||Xw-y||^{2}$$

**Связь линейной регрессии с МНК**

Аналитическое решение задачи (МНК) сводится к решению системы линейных алгебраических уравнений (СЛАУ) 
  $$X^T X w = X^T y$$

**Методы предобработки данных**

- Масштабирование (`StandardScaler`): Приводит признаки к единому масштабу путем вычитания среднего (`with_mean`) и деления на стандартное отклонение (`with_std`). Если этого не сделать, признак с большими числовыми значениями, признак с большими числовыми значениями (например, налоги на дом в тысячах долларов) будет иметь больший вес при обучении, чем признак с маленькими значениями (например, количество комнат), хотя второй может быть важнее.
- Полиномиальные признаки (`PolynomialFeatures`): Создает новые признаки путем возведения исходных в степень `degree` и их перемножения. Это позволяет линейной модели выявлять нелинейные зависимости. Линейная регрессия сама по себе может строить только прямые линии (или гиперплоскости). Чтобы она могла улавливать сложные, нелинейные зависимости в данных, мы искусственно создаем новые признаки.

**Метрики качества**

- MAE (Mean Absolute Error): Средняя абсолютная ошибка, оценивающая разницу между предсказаниями и реальными значениями на обучающей (`train`) и тестовой (`test`) выборках.
- Число обусловленности: Вычисляется для матрицы $X^T X$ и показывает чувствительность точного решения к ошибкам округления. Большие значения указывают на проблемы.



In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# 1.Загрузка данных
df = pd.read_csv('housing.csv', delimiter=';')

# Признаки X
X = df.drop(columns=['median_house_value']).fillna(0)
# Целевая переменная y
y = df['median_house_value']

# Параметры разбения данных (вариант 4)
test_size = 0.27
random_state = 40

# Ручная реализация линейной регрессии
class MatrixLinearRegression:
    def __init__(self):
        self.weights = None
        self.bias = None

    def fit(self, x, y):
        X = np.insert(x, 0, 1, axis=1) # Добавляем столбец единиц для учета свободного члена
        XT_X_inv = np.linalg.inv(X.T @ X) # Вычисляем (X^T * X)**(-1)
        weights = np.linalg.multi_dot([XT_X_inv, X.T, y]) # XT_X_inv @ X.T @ y
        self.bias = weights[0] # Первый элемент - свободный член
        self.weights = weights[1:] # Остальные элементы - коэффициенты при признаках
    def predict(self, X):
        return X @ self.weights + self.bias
    
# Ручная реализация МАЕ
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

# 2. Проведение экспериментов

# a) Разделение данных на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)


# Параметры экспериментов
degrees = [1, 2, 3] 
std_options = [False, True] 
mean_options = [False, True]

experiments = []
# Генерация всех комбинаций параметров для экспериментов
for deg in degrees:
    for std_val in std_options:
        for mean_val in mean_options:
            experiments.append({
                'std': std_val, 
                'mean': mean_val, 
                'degree': deg
            })

results = []

for i, exp in enumerate(experiments, 1):
    # Инициализация препроцессоров
    scaler = StandardScaler(with_std=exp['std'], with_mean=exp['mean'])
    # include_bias=False запрещает добавлять в матрицу признаков нулевую степень (столбец из единиц)
    poly = PolynomialFeatures(degree=exp['degree'], include_bias=False)

    # b) Предобработка данных
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    X_train_poly = poly.fit_transform(X_train_scaled)
    X_test_poly = poly.transform(X_test_scaled)

    # c) Обучение модели
    model = MatrixLinearRegression()
    model.fit(X_train_poly, y_train)
    
    # Предсказание и оценка
    y_train_pred = model.predict(X_train_poly)
    y_test_pred = model.predict(X_test_poly)

    # d) Вычисление метрик
    # Вычисление MAE
    mae_train = mae(y_train, y_train_pred)
    mae_test = mae(y_test, y_test_pred)

    # Вычисление обусловленности матрицы X^T * X
    XT_X = X_train_poly.T @ X_train_poly
    cond_number = np.linalg.cond(XT_X)

    results.append({
        '№ эксперимента': i,
        'std': exp['std'],
        'mean': exp['mean'],
        'degree': exp['degree'],
        'MAE train': mae_train,
        'MAE test': mae_test,
        'Число обусловленности': f'{cond_number:.2e}',
    })

results_df = pd.DataFrame(results)
results_df

,№ эксперимента,std,mean,degree,MAE train,MAE test,Число обусловленности
0,1,False,False,1,50742.589261,51735.985379,8.77e+06
1,2,False,True,1,50742.589260,51735.985379,2.51e+07
2,3,True,False,1,50742.589260,51735.985379,1.47e+05
3,4,True,True,1,50742.589260,51735.985379,1.53e+02
4,5,False,False,2,45119.288547,45969.964951,2.05e+19
5,6,False,True,2,45119.286940,45969.963526,9.68e+15
6,7,True,False,2,45119.282412,45969.959713,8.79e+12
7,8,True,True,2,45119.286940,45969.963526,2.21e+05
8,9,False,False,3,41491.200540,43633.278927,1.22e+26
9,10,False,True,3,41494.556874,43637.394336,7.93e+24


Результаты при использовании готовыз функций
```txt   № эксперимента    std   mean  degree     MAE train      MAE test  \
0                1  False  False       1  50742.589260  51735.985379   
1                2  False   True       1  50742.589260  51735.985379   
2                3   True  False       1  50742.589260  51735.985379   
3                4   True   True       1  50742.589260  51735.985379   
4                5  False  False       2  45119.286919  45969.963414   
5                6  False   True       2  45119.286942  45969.963529   
6                7   True  False       2  45119.286940  45969.963526   
7                8   True   True       2  45119.286940  45969.963526   
8                9  False  False       3  42281.315372  44328.924779   
9               10  False   True       3  52953.934776  54461.330459   
10              11   True  False       3  41494.556959  43637.394359   
11              12   True   True       3  41494.556962  43637.394358  
   Число обусловленности  
0               8.77e+06  
1               2.51e+07  
2               1.47e+05  
3               1.53e+02  
4               2.05e+19  
5               9.68e+15  
6               8.79e+12  
7               2.21e+05  
8               1.22e+26  
9               7.93e+24  
10              2.51e+19  
11              1.91e+09```